In [60]:
from simfile import SimFile
from simulation import Simulation
from validation import switchOffAllEffects
import os
import numpy as np
import referenceFrames as rf
import matplotlib.pyplot as plt
from scipy import constants
from math import radians, degrees

In [61]:
sim = Simulation("StarPosition")
switchOffAllEffects(sim)
sim.outputDir = os.environ["PLATO_WORKDIR"]

sim["ObservingParameters/NumExposures"] = 1
sim["SubField/NumRows"] = 100
sim["SubField/NumColumns"] = 100

# One full-frame exposure

sim["CCD/IncludeConvolution"] = "no"
sim["PSF/Model"] = "MappedGaussian"

pixelSize = sim["CCD/PixelSize"] * constants.micro / constants.milli

In [62]:
# Focal-plane coordinates at the centre of the CCD [mm]

xFP = sim["SubField/NumColumns"] / 2 * pixelSize
yFP = sim["SubField/NumRows"] / 2 * pixelSize

focalLength = sim["Camera/FocalLength/ConstantValue"] / constants.milli
raPlatform = radians(sim["ObservingParameters/RApointing"])
decPlatform = radians(sim["ObservingParameters/DecPointing"])

# Sky coordinates at the centre of the CCD [radians]

ra, dec = rf.focalPlaneToSkyCoordinates(-xFP, yFP, raPlatform, decPlatform, 0, 0, 0, 0, focalLength)

ra = degrees(ra)
dec = degrees(dec)

In [63]:
starCatalogFilename = os.environ["PLATO_WORKDIR"] + "generatedFromPixelCoordinates.starcat"

myFile = open(starCatalogFilename, "w")
myFile.write("# RA DEC Vmag starID\n")
myFile.write("{0}  {1}  {2}  {3}\n".format(ra, dec, 12.5, 1))
myFile.close()

sim["ObservingParameters/StarCatalogFile"] = starCatalogFilename

# dim = 100
# sim["SubField/NumRows"] = dim
# sim["SubField/NumColumns"] = dim
# sim["SubField/ZeroPointRow"] = int(sim["SubField/NumRows"] / 2 - dim / 2)
# sim["SubField/ZeroPointColumn"] = int(sim["SubField/NumColumns"] / 2 - dim / 2)

output = sim.run(removeOutputFile = True)


2019-06-13 13:33:00 WARNING Simulation: no information about detected stars to write to HDF5
2019-06-13 13:33:01 WARNING Camera: No star positions to write to HDF5 file.



In [58]:
print(xFP, yFP)

40.589999999999996 40.589999999999996


In [59]:
print(ra, dec)

145.36112574983323 -66.5915443865982


In [64]:
print(focalLength)

247.51999999999998
